# Healthcare Utilization & Cost Analytics
## 03. Clinical Data Analysis

This notebook examines the clinical tables: observations, conditions, procedures, medications and immunizations. It also resolves the data quality questions left open in `01_data_profiling.ipynb` (observations without an encounter, duplicated observations).

**Questions**
- Why do 10% of observations have no encounter, and are the duplicates a problem?
- Which conditions are the most common, and which are ongoing?
- Do patients with more ongoing conditions use more healthcare?
- Which procedures and medications are the most frequent and the most costly?
- How much of the recorded cost comes from encounters, procedures, medications and immunizations?

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 2. Load Data

Only the tables and columns needed here are loaded. Clinical tables contain patient IDs but no names or addresses.

In [ ]:
data_dir = Path("../data/raw")

patients = pd.read_csv(data_dir / "patients.csv", usecols=["Id"])
encounters = pd.read_csv(data_dir / "encounters.csv", usecols=["Id", "PATIENT", "TOTAL_CLAIM_COST"])
conditions = pd.read_csv(data_dir / "conditions.csv")
procedures = pd.read_csv(data_dir / "procedures.csv")
observations = pd.read_csv(data_dir / "observations.csv", low_memory=False)
medications = pd.read_csv(data_dir / "medications.csv")
immunizations = pd.read_csv(data_dir / "immunizations.csv")

for name, df in [("conditions", conditions), ("procedures", procedures), ("observations", observations),
                 ("medications", medications), ("immunizations", immunizations)]:
    print(f"{name}: {len(df):,} rows")

## 3. Observations: Data Quality

### 3.1 Observation Types

In [ ]:
observations["TYPE"].value_counts()

### 3.2 Observations Without an Encounter

In notebook 01, about 10% of observations had an empty `ENCOUNTER` field. This checks which observations they are.

In [ ]:
no_enc = observations[observations["ENCOUNTER"].isna()]

print(f"Observations without encounter: {len(no_enc):,} ({len(no_enc) / len(observations) * 100:.1f}%)")
print(f"Patients with such observations: {no_enc['PATIENT'].nunique():,} of {observations['PATIENT'].nunique():,}")
print()
print(no_enc["DESCRIPTION"].value_counts())

# Are there observations without an encounter that are NOT one of these three?
other = no_enc[~no_enc["DESCRIPTION"].isin(["QALY", "DALY", "QOLS"])]
print("\nOther observations without encounter:", len(other))

The missing encounters belong to three descriptions (`QALY`, `DALY`, `QOLS`). These are patient-level quality-of-life indicators, not measurements taken during a visit, so an empty `ENCOUNTER` is expected and not a data error.

### 3.3 Duplicated Observations

In [ ]:
dup_mask = observations.duplicated()
print("Duplicated rows:", dup_mask.sum())

observations[dup_mask]["DESCRIPTION"].value_counts().head(10)

In [ ]:
# Remove exact duplicates for all further analysis
observations = observations.drop_duplicates()
print(f"Observations after removing duplicates: {len(observations):,}")

### 3.4 Missing Units

In [ ]:
missing_units = observations[observations["UNITS"].isna()]
print(f"Observations without units: {len(missing_units):,}")
missing_units["TYPE"].value_counts()

Missing units occur only for text observations, which have no unit by nature.

### 3.5 Most Common Measurements

In [ ]:
measured = observations[observations["ENCOUNTER"].notna() & (observations["TYPE"] == "numeric")]
measured["DESCRIPTION"].value_counts().head(10)

## 4. Conditions

### 4.1 Overview

In [ ]:
print(f"Condition records:            {len(conditions):,}")
print(f"Patients with any condition:   {conditions['PATIENT'].nunique():,} of {len(patients):,}")
print(f"Different condition names:     {conditions['DESCRIPTION'].nunique()}")
print(f"Records without end date:      {conditions['STOP'].isna().mean() * 100:.1f}%")

conditions_per_patient = conditions.groupby("PATIENT").size()
print("\nConditions per patient:")
conditions_per_patient.describe().round(1)

### 4.2 Condition Types

The text in brackets at the end of a condition name tells what kind of record it is. Only `disorder` records are diagnoses, so the following analysis uses those.

In [ ]:
conditions["TYPE"] = conditions["DESCRIPTION"].str.extract(r"\(([^()]+)\)$")[0].fillna("no suffix")
conditions["TYPE"].value_counts()

### 4.3 Most Common Disorders

Prevalence is the share of all patients who ever had the condition.

In [ ]:
disorders = conditions[conditions["TYPE"] == "disorder"].copy()

prevalence = (
    disorders.groupby("DESCRIPTION")["PATIENT"].nunique().sort_values(ascending=False).head(15)
    / len(patients) * 100
).round(1)

prevalence.sort_values().plot(kind="barh", figsize=(9, 6), title="Most Common Disorders (% of patients)")
plt.xlabel("Share of patients (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

prevalence

### 4.4 Ongoing Disorders

A disorder without an end date (`STOP` is empty) is treated as ongoing.

In [ ]:
ongoing = disorders[disorders["STOP"].isna()]

ongoing_prevalence = (
    ongoing.groupby("DESCRIPTION")["PATIENT"].nunique().sort_values(ascending=False).head(10)
    / len(patients) * 100
).round(1)

ongoing_prevalence

### 4.5 Ongoing Disorders and Encounters

Do patients with more ongoing disorders have more encounters? This shows an association only. Older patients have both more disorders and more encounters, so it does not prove that disorders cause the visits.

In [ ]:
ongoing_per_patient = ongoing.groupby("PATIENT")["DESCRIPTION"].nunique().rename("ongoing_disorders")
enc_per_patient = encounters.groupby("PATIENT").size().rename("encounters")

combined = pd.concat([enc_per_patient, ongoing_per_patient], axis=1)
combined["ongoing_disorders"] = combined["ongoing_disorders"].fillna(0)
combined["group"] = pd.cut(
    combined["ongoing_disorders"], bins=[-1, 0, 2, 5, 100], labels=["0", "1-2", "3-5", "6+"]
)

summary = combined.groupby("group", observed=True)["encounters"].agg(["count", "median", "mean"]).round(1)
summary.columns = ["patients", "median_encounters", "mean_encounters"]
summary

In [ ]:
summary["median_encounters"].plot(kind="bar", figsize=(6, 4), title="Median Encounters by Number of Ongoing Disorders")
plt.xlabel("Ongoing disorders per patient")
plt.ylabel("Median encounters")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Procedures

### 5.1 Overview

In [ ]:
print(f"Procedures: {len(procedures):,}")
print(f"Encounters with at least one procedure: {procedures['ENCOUNTER'].nunique():,} "
      f"of {len(encounters):,} ({procedures['ENCOUNTER'].nunique() / len(encounters) * 100:.1f}%)")
print(f"Median cost: ${procedures['BASE_COST'].median():,.0f} | mean: ${procedures['BASE_COST'].mean():,.0f} "
      f"| max: ${procedures['BASE_COST'].max():,.0f}")

### 5.2 Most Frequent and Most Costly Procedures

In [ ]:
proc_summary = procedures.groupby("DESCRIPTION").agg(
    procedures=("CODE", "size"),
    total_cost=("BASE_COST", "sum"),
    avg_cost=("BASE_COST", "mean"),
)

print("Top 10 by number of procedures")
display(proc_summary.sort_values("procedures", ascending=False).head(10).round(0))

print("Top 10 by total cost")
display(proc_summary.sort_values("total_cost", ascending=False).head(10).round(0))

Costs in Synthea are generated, not taken from a real price list. Some of them look unrealistic (for example routine checks priced in the thousands), so these figures describe the synthetic dataset only and should not be read as real procedure prices.

## 6. Medications

### 6.1 Overview

In [ ]:
print(f"Medication records: {len(medications):,}")
print(f"Patients with any medication: {medications['PATIENT'].nunique():,} of {len(patients):,}")
print(f"Different medications: {medications['DESCRIPTION'].nunique()}")
print(f"Total cost: ${medications['TOTALCOST'].sum():,.0f}")
print(f"Paid by payers: ${medications['PAYER_COVERAGE'].sum():,.0f} "
      f"({medications['PAYER_COVERAGE'].sum() / medications['TOTALCOST'].sum() * 100:.1f}%)")

### 6.2 Most Costly Medications

In [ ]:
med_summary = medications.groupby("DESCRIPTION").agg(
    prescriptions=("CODE", "size"),
    patients=("PATIENT", "nunique"),
    total_cost=("TOTALCOST", "sum"),
    payer_coverage=("PAYER_COVERAGE", "sum"),
)
med_summary["coverage_pct"] = (med_summary["payer_coverage"] / med_summary["total_cost"] * 100).round(1)

med_summary.sort_values("total_cost", ascending=False).head(10).round({"total_cost": 0, "payer_coverage": 0, "coverage_pct": 1})

## 7. Recorded Cost by Source

Adds up the cost fields of the four tables that carry a cost. The tables record costs separately, so this is a comparison of the sources, not an exact total of claims.

In [ ]:
cost_by_source = pd.Series({
    "Encounters": encounters["TOTAL_CLAIM_COST"].sum(),
    "Procedures": procedures["BASE_COST"].sum(),
    "Medications": medications["TOTALCOST"].sum(),
    "Immunizations": immunizations["BASE_COST"].sum(),
}).sort_values(ascending=False)

cost_table = cost_by_source.to_frame("total_cost")
cost_table["share_pct"] = (cost_table["total_cost"] / cost_table["total_cost"].sum() * 100).round(1)
display(cost_table.round({"total_cost": 0, "share_pct": 1}))

(cost_by_source / 1e6).plot(kind="bar", figsize=(7, 4), title="Recorded Cost by Source")
plt.ylabel("Million USD")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Key Findings

**Data quality**
- **Observations without encounter:** 30,363 observations (10.1%) have no `ENCOUNTER`. All of them are `QALY`, `DALY` or `QOLS` (10,121 each), quality-of-life summary measures recorded for 1,056 of the 1,171 patients. They are not tied to a visit, so the empty field is expected and not an error.
- **Duplicates:** 442 exact duplicate observations, almost all of them routine laboratory values (calcium, potassium, glucose, urea nitrogen, sodium, chloride, carbon dioxide; 62-63 each) plus 6 pain scores. They were removed, leaving 299,255 observations.
- **Missing units:** All 12,735 observations without units are text observations, which have no unit by nature.

**Conditions**
- 8,376 condition records cover 1,152 of 1,171 patients (98.4%); the median patient has 7. 45.5% of records have no end date. 3,897 records are disorders (diagnoses); the rest are findings, situations and records without a type label.
- The most common disorders are viral sinusitis (63.5% of patients), acute viral pharyngitis (42.0%), acute bronchitis (39.6%), anemia (25.6%) and chronic sinusitis (19.9%). The most common ongoing disorders are anemia (23.6%) and chronic sinusitis (19.6%), followed by hypertriglyceridemia and metabolic syndrome X (6.3% each).
- Patients with more ongoing disorders have more encounters: the median is 21 for patients with none, 33 for 1-2, 60 for 3-5 and 114 for 6 or more. This is an association only, as age affects both.

**Procedures and medications**
- 34,981 procedures took place in 39.5% of encounters. The most frequent is medication reconciliation (5,632); the costliest in total is electrical cardioversion ($31.1M for 981 procedures). Synthea costs are generated, so these are not real prices.
- 42,989 medication records for 1,107 patients cost $100.1M in total. Payers covered only 6.3%, far below the 48.2% coverage of encounter claims. The costliest medication is simvastatin 10 MG ($13.4M, 2% covered).

**Cost by source**
- Procedures account for 61.2% of recorded cost, medications for 35.5%, encounters for 2.4% and immunizations for 0.8%. The encounter claim costs analyzed in notebook 02 are therefore only a small part of recorded costs, and the payer coverage found there describes encounter claims, not total spending.